# 01 · Data Profiling

**Goal:** Understand the raw data shape — descriptive statistics only, no interpretation.

**Output:** Reference numbers used to build rules for `validation.py` and `data_config.yaml`

---

### 📑 Table of Contents

| Section | Content |
|:---:|---|
| 0 | Setup & Load |
| 1 | Shape & Schema |
| 2 | Missing Values |
| 3 | Descriptive Statistics |
| 4 | Categorical Columns |
| 5 | Duplicates |
| 6 | Reference Summary → `data_config.yaml` |
| 7 | Save Profiling Report (Optional) |

---

> ⚠️ **Important:** This notebook is **read-only**. It does NOT modify any data.
> All numbers here will be manually copied to `configs/data_config.yaml` after EDA.

## 0 · Setup & Load

In [17]:
# ===================================================================
# Section 0 · Setup
# ===================================================================
import os
import sys
from pathlib import Path

from src.utils.paths import PROJECT_DIR

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

✅ Working dir : /content/california_housing_full_project
✅ sys.path[0] : /content/california_housing_full_project


In [3]:
# Load the dataset using the project's DataLoader
import pandas as pd
import numpy as np

from src.data.data_loader import DataLoader

# Initialize the DataLoader
loader = DataLoader()

# Load the raw data fetched by DVC (or Kaggle fallback)
df = loader.load_raw("housing.csv")

# --- NEW: Safety Check ---
# This prevents cryptic errors if the user forgot to run 00_environment_setup.ipynb
if df is None or df.empty:
    raise RuntimeError(
        "❌ Data not found or empty. Please ensure you have run "
        "'00_environment_setup.ipynb' successfully to execute `dvc pull`."
    )

print(f"✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

2026-09-01 02:02:23 | INFO     | numexpr.utils | NumExpr defaulting to 2 threads.
2026-09-01 02:02:23 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-09-01 02:02:23 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/raw/housing.csv
2026-09-01 02:02:23 | INFO     | src.data.data_loader | Loaded 'housing.csv' | shape=(20640, 10) | stage=raw
✅ Loaded: 20,640 rows × 10 columns


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


---
##  1 · Shape & Schema

> **Question:** How many rows? How many columns? What is the type of each column?


In [4]:
# ===================================================================
# Section 1 · Shape & Schema
# ===================================================================
print("="*50)
print(f"  Rows    : {df.shape[0]:>10,}")
print(f"  Columns : {df.shape[1]:>10}")
print("="*50)

  Rows    :     20,640
  Columns :         10


In [5]:
schema = pd.DataFrame({
    "dtype"    : df.dtypes,
    "non_null" : df.count(),
    "null"     : df.isnull().sum(),
    "unique"   : df.nunique(),
})

schema["null_%"] = (schema["null"] / len(df) * 100).round(2)
print("Schema Overview:")
schema

Schema Overview:


,dtype,non_null,null,unique,null_%
longitude,float64,20640,0,844,0.0
latitude,float64,20640,0,862,0.0
housing_median_age,float64,20640,0,52,0.0
total_rooms,float64,20640,0,5926,0.0
total_bedrooms,float64,20433,207,1923,1.0
population,float64,20640,0,3888,0.0
households,float64,20640,0,1815,0.0
median_income,float64,20640,0,12928,0.0
median_house_value,float64,20640,0,3842,0.0
ocean_proximity,object,20640,0,5,0.0


---
## 2 · Missing Values

> **Question:** Where are the nulls? What percentage? Are they acceptable?
> 
> **Threshold:** < 5% is acceptable for imputation.

In [6]:
# ===================================================================
# Section 2 · Missing Values
# ===================================================================
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

if nulls.empty:
    print("✅ No missing values found.")
else:
    null_report = pd.DataFrame({
        "null_count" : nulls,
        "null_%"     : (nulls / len(df) * 100).round(3),
        "threshold"  : "< 5%",
        "status"     : (nulls / len(df) * 100).apply(
                            lambda x: "✅ OK" if x < 5 else "⚠️ HIGH"
                        )
    })
    print("Columns with missing values:")
    display(null_report)

Columns with missing values:


,null_count,null_%,threshold,status
total_bedrooms,207,1.003,< 5%,✅ OK


---
## 3 · Descriptive Statistics

> **Question:** What are the min/max/mean/std for each numeric column?
> 
> ⚠️ Here we only **record** the numbers — interpretation comes in EDA.

In [7]:
# ===================================================================
# Section 3 · Descriptive Statistics
# ===================================================================
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
longitude,20640.0,-119.570,2.004,-124.35,-121.800,-118.490,-118.010,-114.31
latitude,20640.0,35.632,2.136,32.54,33.930,34.260,37.710,41.95
housing_median_age,20640.0,28.639,12.586,1.00,18.000,29.000,37.000,52.00
total_rooms,20640.0,2635.763,2181.615,2.00,1447.750,2127.000,3148.000,39320.00
total_bedrooms,20433.0,537.871,421.385,1.00,296.000,435.000,647.000,6445.00
population,20640.0,1425.477,1132.462,3.00,787.000,1166.000,1725.000,35682.00
households,20640.0,499.540,382.330,1.00,280.000,409.000,605.000,6082.00
median_income,20640.0,3.871,1.900,0.50,2.563,3.535,4.743,15.00
median_house_value,20640.0,206855.817,115395.616,14999.00,119600.000,179700.000,264725.000,500001.00


In [8]:
# Reference boundaries per column (to be placed in data_config.yaml)
numeric_cols = df.select_dtypes(include="number").columns

boundaries = pd.DataFrame({
    "min"    : df[numeric_cols].min(),
    "max"    : df[numeric_cols].max(),
    "mean"   : df[numeric_cols].mean().round(3),
    "median" : df[numeric_cols].median(),
    "std"    : df[numeric_cols].std().round(3),
    "q1"     : df[numeric_cols].quantile(0.25),
    "q3"     : df[numeric_cols].quantile(0.75),
})

print("Numeric Boundaries (Reference for data_config.yaml):")
boundaries

Numeric Boundaries (Reference for data_config.yaml):


,min,max,mean,median,std,q1,q3
longitude,-124.3500,-114.3100,-119.570,-118.4900,2.004,-121.8000,-118.01000
latitude,32.5400,41.9500,35.632,34.2600,2.136,33.9300,37.71000
housing_median_age,1.0000,52.0000,28.639,29.0000,12.586,18.0000,37.00000
total_rooms,2.0000,39320.0000,2635.763,2127.0000,2181.615,1447.7500,3148.00000
total_bedrooms,1.0000,6445.0000,537.871,435.0000,421.385,296.0000,647.00000
population,3.0000,35682.0000,1425.477,1166.0000,1132.462,787.0000,1725.00000
households,1.0000,6082.0000,499.540,409.0000,382.330,280.0000,605.00000
median_income,0.4999,15.0001,3.871,3.5348,1.900,2.5634,4.74325
median_house_value,14999.0000,500001.0000,206855.817,179700.0000,115395.616,119600.0000,264725.00000


---
## 4 · Categorical Columns

> **Question:** What are the unique categories and their distribution?

In [9]:
# ===================================================================
# Section 4 · Categorical Columns
# ===================================================================
cat_cols = df.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(2)
    summary = pd.DataFrame({"count": vc, "%": pct})

    print(f"\n{'='*45}")
    print(f"  Column : {col}")
    print(f"  Unique : {df[col].nunique()}")
    print(f"{'='*45}")
    display(summary)


  Column : ocean_proximity
  Unique : 5


,count,%
ocean_proximity,,
<1H OCEAN,9136,44.26
INLAND,6551,31.74
NEAR OCEAN,2658,12.88
NEAR BAY,2290,11.09
ISLAND,5,0.02


---
## 5 · Duplicates

> **Question:** Are there any duplicate rows?

In [10]:
# ===================================================================
# Section 5 · Duplicates
# ===================================================================
n_dup = df.duplicated().sum()
pct   = round(n_dup / len(df) * 100, 3)

print(f"Duplicate rows : {n_dup:,}  ({pct}%)")

if n_dup > 0:
    print("\nSample duplicates:")
    display(df[df.duplicated(keep=False)].head(6))
else:
    print("✅ No duplicate rows.")

Duplicate rows : 0  (0.0%)
✅ No duplicate rows.


---
## 6 · Reference Summary → `data_config.yaml`

> Here we print the final numbers that should be **manually placed** in `configs/data_config.yaml`.
> This is the **official output** of Profiling.
> 
> ⚠️ **Not saved automatically** — the decision is yours after EDA.

In [11]:
# ===================================================================
# Section 6 · Reference Summary → data_config.yaml
# ===================================================================
print("\n" + "="*60)
print("  PROFILING REFERENCE SUMMARY")
print("  Copy relevant values → configs/data_config.yaml")
print("="*60)

print(f"\ndataset:")
print(f"  rows    : {len(df):,}")
print(f"  columns : {df.shape[1]}")

print(f"\nmissing_values:")
for col, n in df.isnull().sum().items():
    pct = round(n / len(df) * 100, 3)
    print(f"  {col}: {n} ({pct}%)")

print(f"\nnumeric_boundaries:")
for col in df.select_dtypes(include="number").columns:
    print(f"  {col}:")
    print(f"    min: {df[col].min()}")
    print(f"    max: {df[col].max()}")
    print(f"    mean: {round(df[col].mean(), 3)}")

print(f"\ncategorical_values:")
for col in df.select_dtypes(include="object").columns:
    vals = df[col].unique().tolist()
    print(f"  {col}: {vals}")

print(f"\nduplicates: {df.duplicated().sum()}")
print("="*60)

# NEW: Helpful reminder for the user
print("\n📝 Remember to manually copy the 'numeric_boundaries' and 'categorical_values'")
print("   into 'configs/data_config.yaml' under the 'validation' section.\n")


  PROFILING REFERENCE SUMMARY
  Copy relevant values → configs/data_config.yaml

dataset:
  rows    : 20,640
  columns : 10

missing_values:
  longitude: 0 (0.0%)
  latitude: 0 (0.0%)
  housing_median_age: 0 (0.0%)
  total_rooms: 0 (0.0%)
  total_bedrooms: 207 (1.003%)
  population: 0 (0.0%)
  households: 0 (0.0%)
  median_income: 0 (0.0%)
  median_house_value: 0 (0.0%)
  ocean_proximity: 0 (0.0%)

numeric_boundaries:
  longitude:
    min: -124.35
    max: -114.31
    mean: -119.57
  latitude:
    min: 32.54
    max: 41.95
    mean: 35.632
  housing_median_age:
    min: 1.0
    max: 52.0
    mean: 28.639
  total_rooms:
    min: 2.0
    max: 39320.0
    mean: 2635.763
  total_bedrooms:
    min: 1.0
    max: 6445.0
    mean: 537.871
  population:
    min: 3.0
    max: 35682.0
    mean: 1425.477
  households:
    min: 1.0
    max: 6082.0
    mean: 499.54
  median_income:
    min: 0.4999
    max: 15.0001
    mean: 3.871
  median_house_value:
    min: 14999.0
    max: 500001.0
    mean: 20

---
## 7 · Save Profiling Report (Optional)

> Save the profiling results as a Markdown file in `reports/profiling/`.
> This is useful for documentation and tracking changes over time.
> 
> ⚠️ This directory is **NOT tracked by DVC**, so it won't interfere with `dvc status`.

In [15]:
# ===================================================================
# OPTIONAL: Save Profiling Report to File
# ===================================================================
# This report is saved in 'reports/profiling/'.
# Important: This directory is NOT tracked by DVC, so it will not
# interfere with 'dvc status' or 'dvc repro'.
# ===================================================================
from pathlib import Path
from datetime import datetime

# Generate a consistent timestamp
now = datetime.now()
timestamp_str = now.strftime("%Y%m%d_%H%M%S")
iso_timestamp = now.isoformat()

# Define output directory
reports_dir = Path("reports/profiling")
reports_dir.mkdir(parents=True, exist_ok=True)

report_path = reports_dir / f"profiling_report_{timestamp_str}.md"

with open(report_path, "w") as f:
    # Header
    f.write("# Data Profiling Report\n\n")
    f.write(f"**Timestamp**: {iso_timestamp}\n\n")
    
    # 1. Dataset Summary
    f.write("## Dataset Summary\n\n")
    f.write(f"- **Rows**: {len(df):,}\n")
    f.write(f"- **Columns**: {df.shape[1]}\n\n")
    
    # 2. Missing Values
    f.write("## Missing Values\n\n")
    null_counts = df.isnull().sum()
    if null_counts.sum() == 0:
        f.write("✅ No missing values detected.\n\n")
    else:
        for col, n in null_counts.items():
            if n > 0:
                pct = round(n / len(df) * 100, 3)
                f.write(f"- **{col}**: {n} ({pct}%)\n")
        f.write("\n")
    
    # 3. Categorical Columns
    cat_cols = df.select_dtypes(include="object").columns
    if not cat_cols.empty:
        f.write("## Categorical Columns\n\n")
        for col in cat_cols:
            unique_vals = df[col].unique().tolist()
            f.write(f"- **{col}**: {len(unique_vals)} unique values → {unique_vals}\n")
        f.write("\n")
    
    # 4. Numeric Boundaries
    f.write("## Numeric Boundaries\n\n")
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        f.write(f"### {col}\n")
        f.write(f"- **min**: {df[col].min()}\n")
        f.write(f"- **max**: {df[col].max()}\n")
        f.write(f"- **mean**: {round(df[col].mean(), 3)}\n")
        f.write("\n")

try:
    print(f"✅ Profiling report saved → {report_path}")
except Exception as e:
    print(f"⚠️ Failed to save report: {e}")

✅ Profiling report saved → reports/profiling/profiling_report_20260901_021652.md
